# 12 · Grelha taxa × épocas × τ no BERTimbau e no Albertina

Spike E5-S1 do [`docs/backlog.md`](../docs/backlog.md). Os números que saem daqui vão para o [`docs/qualidade_dados.md`](../docs/qualidade_dados.md) (secção 23+, a escrever).

**A pergunta:** para cada modelo tunado, que combinação de **taxa de aprendizagem**, **época** e **τ** dá o melhor top-5 no código completo da validação?

Os três termos, com um exemplo:

- **Taxa de aprendizagem:** o tamanho do passo com que os pesos mudam em cada lote. Se um peso vale 0,50 e o gradiente diz "desce", com taxa 2e-5 (0,00002) o passo é minúsculo; com 5e-5 é 2,5 vezes maior. Grande demais, o modelo esquece o português que já sabia; pequena demais, aprende pouco em 3 épocas.
- **Época k:** o modelo gravado no fim da época k de um treino de 3 épocas. **Não** é igual a um treino de só k épocas, porque a taxa desce ao longo das 3 (descida linear): na época 1 de um treino de 3 a taxa ainda vai alta.
- **τ:** a correção das etiquetas frequentes (Menon et al., 2021). À pontuação de cada etiqueta soma-se −τ × log(frequência no treino). Com τ = 0,5, uma etiqueta que é 10% do treino ganha 0,5 × 2,30 ≈ 1,15 pontos, e uma que é 0,01% ganha 0,5 × 9,21 ≈ 4,61: as raras sobem mais, por isso as frequentes descem em relação a elas. τ = 0 é sem correção. Só reordena as pontuações gravadas, por isso não pede treino novo.

Uma **opção** é um trio (taxa, época, τ). São 3 taxas × 3 épocas × 3 τ = **27 opções por modelo**, mas só **3 treinos** na GPU por modelo: a época e o τ saem do mesmo treino. O N dos códigos raros fica fixo em 10 (limitação escrita no E5-S1: o N não é testado).

**Como:** as opções, a opção em vigor, a tolerância do controlo e as sementes estão em [`experiencias/e5_s1.yaml`](../experiencias/e5_s1.yaml), escritas **antes** de correr; este notebook lê esse ficheiro, para não poder decidir com uma lista diferente da que se treinou. Os treinos correm num pod com GPU e as pastas copiam-se para `metrics/e5_s1/`. O teste não se abre aqui.

**Lição que vale fora do CPV:** quando a mesma regra escolhe entre muitas opções, o número do vencedor fica otimista (**viés de seleção**). Com 27 moedas atiradas, alguma dá mais caras só por sorte. Por isso a regra está escrita antes, o vencedor tem de bater a opção em vigor com uma diferença que conta (bootstrap) e o número confirma-se no teste, na fase 5.

In [ ]:
import json
import os
import sys
from pathlib import Path

import pandas as pd
import yaml
from IPython.display import display

sys.path.insert(0, str(Path.cwd().parent / "src"))
import comparison as cmp
import evaluation as ev
import grelha_bert as gb
from cpv_data import file_sha256

RAIZ = Path.cwd().parent
DATA = RAIZ / "data" / "processed"
# Só para ensaiar o notebook noutra árvore de pastas (por defeito, as do repositório): o YAML e a
# pasta contra a qual se leem os caminhos do YAML (metrics/e5_s1/..., metrics/bert/...).
YAML = Path(os.environ.get("GRELHA_BERT_YAML", RAIZ / "experiencias" / "e5_s1.yaml"))
RAIZ_METRICS = Path(os.environ.get("GRELHA_BERT_RAIZ", RAIZ))


def ler_experiencia(path):
    # Todas as chaves do YAML leem-se aqui e só aqui: se o YAML mudar de nome a uma chave, muda-se
    # esta função. Devolve um dicionário com nomes fixos para o resto do notebook.
    y = yaml.safe_load(Path(path).read_text(encoding="utf-8"))
    g, vig, ctl, sem = y["grelha"], y["em_vigor"], y["controlo"], y.get("sementes", {})
    n_min = g["n_min"][0] if isinstance(g["n_min"], list) else g["n_min"]
    exp = {
        "item": y["item"],
        "saida": y["saida"],
        "modelos": {m: {"primeiro_treino": d["primeiro_treino"],
                        "top5_primeiro": float(d["top5_codigo_primeiro_treino"])}
                    for m, d in y["modelos"].items()},
        "taxas": [float(t) for t in g["taxas_aprendizagem"]],
        "n_min": int(n_min),
        "epocas_max": int(g["epocas_max"]),
        "epocas": [int(e) for e in g.get("epocas_avaliadas", range(1, int(g["epocas_max"]) + 1))],
        "taus": [float(t) for t in g["taus"]],
        "em_vigor": (float(vig.get("taxa_aprendizagem", vig.get("taxa"))), int(vig["epoca"]), float(vig["tau"])),
        "tolerancia": float(ctl.get("tolerancia_pontos_percentuais", ctl.get("tolerancia_pontos"))),
        "semente_grelha": int(sem.get("grelha", 0)),
        "sementes_extra": [int(s) for s in sem.get("vencedor", sem.get("extra", []))],
        # (modelo, taxa, semente) -> pasta, como o YAML a escreve (as sementes extra não estão lá)
        "pastas": {(t["modelo"], float(t["taxa"]), int(t["semente"])): t["saida"] for t in y.get("treinos", [])},
    }
    for v, nome in ((exp["em_vigor"][0], "taxa"), (exp["em_vigor"][1], "época"), (exp["em_vigor"][2], "τ")):
        lista = {"taxa": exp["taxas"], "época": exp["epocas"], "τ": exp["taus"]}[nome]
        if v not in lista:
            raise ValueError(f"a opção em vigor tem {nome} {v}, que não está na grelha {lista}")
    return exp


EXP = ler_experiencia(YAML)
MODELOS, TAXAS, EPOCAS, TAUS = list(EXP["modelos"]), EXP["taxas"], EXP["epocas"], EXP["taus"]
EM_VIGOR = EXP["em_vigor"]


def pasta(modelo, taxa, semente):
    # A pasta escrita no YAML; para as sementes extra, o mesmo padrão de nome
    rel = EXP["pastas"].get((modelo, taxa, semente)) or gb.pasta_treino(EXP["saida"], modelo, taxa, semente)
    return RAIZ_METRICS / rel


def nome_opcao(o):
    return f"taxa {gb.taxa_txt(o[0])} · época {o[1]} · τ {o[2]:g}"


val = pd.read_parquet(DATA / "validacao.parquet", columns=["idcontrato", "codigos", "visto_no_treino"])
VAL_SHA = file_sha256(DATA / "validacao.parquet")
print(f"YAML: {YAML.relative_to(RAIZ) if YAML.is_relative_to(RAIZ) else YAML} ({EXP['item']})")
print(f"validação: {len(val):,} contratos ({(~val['visto_no_treino']).sum():,} com texto novo)")
print(f"modelos: {MODELOS} · taxas: {[gb.taxa_txt(t) for t in TAXAS]} · épocas: {EPOCAS} · τ: {TAUS} · N = {EXP['n_min']}")
print(f"opções por modelo: {len(TAXAS) * len(EPOCAS) * len(TAUS)} (de {len(TAXAS)} treinos)")
print(f"em vigor: {nome_opcao(EM_VIGOR)} · tolerância do controlo: {EXP['tolerancia']:g} pontos")

## Os treinos

Uma linha por treino, lida do `treino.json` de cada pasta. Antes de ler qualquer sugestão, a célula confirma (com `grelha_bert.verificar_treino`) que cada pasta é o treino que o YAML pediu: o modelo, a taxa, a semente 0, as 3 épocas, o N = 10 e os três τ; que o treino é da janela de 2022 e acaba antes da validação; que nem o treino nem a validação foram o `teste.parquet`; e que o SHA-256 (a impressão digital do ficheiro) da validação do pod é igual ao do portátil. Se faltar uma pasta, pára com erro: as 27 opções só se comparam completas.

**Custo:** horas de GPU = `tempo_treinar_s` / 3600 (só o treino, sem instalação nem cópias, como no E5-T4 e no E5-T5). O preço por hora não vem no `treino.json`: escreve-se à mão no `pod_info.json`, na pasta do treino ou em `metrics/e5_s1/`. Sem ele, os dólares ficam vazios (NaN).

In [ ]:
info, custos, avisos = {}, {}, []
for m in MODELOS:
    for taxa in TAXAS:
        p = pasta(m, taxa, EXP["semente_grelha"])
        if not (p / gb.TREINO_JSON).exists():
            raise FileNotFoundError(f"falta o treino {p.relative_to(RAIZ_METRICS)} (sem {gb.TREINO_JSON}): "
                                    "copia-o do pod antes de correr este notebook")
        resumo = json.loads((p / gb.TREINO_JSON).read_text(encoding="utf-8"))
        for a in gb.verificar_treino(resumo, p.name, modelo=m, taxa=taxa, semente=EXP["semente_grelha"],
                                     epocas=EXP["epocas_max"], n_min=EXP["n_min"], taus=TAUS,
                                     validacao_sha256=VAL_SHA):
            avisos.append(f"{p.name}: {a}")
        info[m, taxa] = resumo
        custos[m, taxa] = gb.custo(resumo, gb.ler_pod_info(p))

treinos = pd.DataFrame({f"{m} · taxa {gb.taxa_txt(t)}": {
    "GPU": c["gpu"], "horas de GPU": c["horas_gpu"], "preço ($/h)": c["preco_por_hora_usd"], "custo ($)": c["dolares"],
    "pico memória GPU (MB)": info[m, t].get("memoria_gpu_mb", {}).get("pico_reservada"),
    "etiquetas": info[m, t].get("etiquetas")} for (m, t), c in custos.items()}).T
display(treinos.style.format({"horas de GPU": "{:.2f}", "preço ($/h)": "{:.2f}", "custo ($)": "{:.2f}"}, na_rep="NaN"))
print(f"total: {treinos['horas de GPU'].sum():.2f} h de GPU · {treinos['custo ($)'].sum(min_count=len(treinos)):.2f} $")
if treinos["custo ($)"].isna().any():
    print("AVISO: falta o preço por hora (pod_info.json) de algum treino: o custo em dólares fica NaN.")
for a in avisos:
    print("AVISO:", a)
print("ok: os treinos são os do YAML")

## As sugestões

Um ficheiro por opção: `epoca_<k>/sugestoes_validacao.parquet` (τ = 0), `..._tau0.25.parquet` e `..._tau0.5.parquet` dentro de cada treino. As verificações são as dos notebooks 04 a 10: uma linha por contrato da validação, alinhada pelo `idcontrato` (nunca pela posição), com 5 códigos distintos de 8 dígitos (`grelha_bert.ler_sugestoes`, que usa `raise` e não `assert`, porque o Python ignora os `assert` quando corre com `-O`).

A **perda na validação** de cada época (E5-T7, a entropia cruzada média nos contratos da validação cuja etiqueta existe no treino) lê-se do `epoca.json`. É só informação: quem decide é o top-5.

In [ ]:
por = {m: {(t, ep, tau): gb.ler_sugestoes(gb.ficheiro_sugestoes(pasta(m, t, EXP["semente_grelha"]), ep, tau), val["idcontrato"])
           for t in TAXAS for ep in EPOCAS for tau in TAUS} for m in MODELOS}
perdas = {m: {(t, ep): gb.perda_validacao(pasta(m, t, EXP["semente_grelha"]), ep) for t in TAXAS for ep in EPOCAS}
          for m in MODELOS}
for m in MODELOS:
    print(f"{m}: {len(por[m])} opções lidas · ok")
if any(pd.isna(v) for d in perdas.values() for v in d.values()):
    print("AVISO: falta a perda na validação em algum epoca.json (treino anterior ao E5-T7?): fica NaN.")

## Controlo: o treino de taxa 2e-5 repete o primeiro treino?

A grelha inclui a configuração do primeiro treino de cada modelo (E5-T4 para o BERTimbau, E5-T5 para o Albertina): taxa 2e-5, 3 épocas, semente 0. Esse treino **repete-se**. Numa GPU, a mesma semente pode não dar exatamente os mesmos números (algumas operações somam em ordem diferente de cada vez), e o bootstrap não mede isso.

O controlo passa se o top-5 no código completo da repetição (τ = 0,25, última época) ficar a menos da **tolerância** do YAML (em pontos percentuais) do primeiro treino. Exemplo: com 76,98% e tolerância 0,5, passa de 76,48% a 77,48%. **Se não passar, não se tira nenhuma conclusão sobre a grelha** antes de a diferença estar explicada; o notebook continua a correr, para se ver o resto, mas a decisão fica suspensa.

Mostra-se também a percentagem de contratos com as 5 sugestões **iguais** às do primeiro treino: 100% quer dizer repetição exata; abaixo disso, a GPU não repetiu bit a bit (o que é normal), e o que conta é o top-5.

In [ ]:
controlo = {}
for m in MODELOS:
    opcao = (EM_VIGOR[0], EXP["epocas_max"], EM_VIGOR[2])  # o treino de taxa 2e-5, última época, τ 0,25
    rep = ev.report(por[m][opcao], val["codigos"], val["visto_no_treino"])
    valor = 100 * rep.loc[("todos", "codigo"), "top5"]
    ref = EXP["modelos"][m]["top5_primeiro"]
    passa, dif = gb.controlo(valor, ref, EXP["tolerancia"])
    controlo[m] = passa
    primeiro = RAIZ_METRICS / EXP["modelos"][m]["primeiro_treino"] / gb.suggestions_file(EM_VIGOR[2])
    iguais = (gb.ler_sugestoes(primeiro, val["idcontrato"]) == por[m][opcao]).mean() if primeiro.exists() else float("nan")
    print(f"{m}: repetição {valor:.2f}% · primeiro treino {ref:.2f}% · diferença {dif:+.2f} pontos "
          f"(tolerância {EXP['tolerancia']:g}) · sugestões iguais: {iguais:.2%}")
    print(f"   -> controlo {'PASSA' if passa else 'FALHA'}")

FALHOU = [m for m, ok in controlo.items() if not ok]
if FALHOU:
    print("\n" + "!" * 100)
    print(f"!!! CONTROLO FALHOU em {FALHOU}: NÃO se tira nenhuma conclusão sobre a grelha destes modelos")
    print("!!! antes de a diferença estar explicada (E5-S1). As células abaixo correm só para se ver o resto.")
    print("!" * 100)

## As 27 opções de cada modelo

Uma tabela por modelo, ordenada pelo top-5 no código completo. Além das métricas da regra (top-5 no código completo e na classe, em todos os contratos), mostram-se o top-5 no texto novo (contratos cujo texto não está no treino: medem o que o modelo generaliza, não o que decorou), o top-1, a perda na validação da época e o custo do treino de onde a opção sai. O custo é o do treino **inteiro** de 3 épocas: para repetir a época 1 de um treino de 3 é preciso o mesmo treino (a taxa desce em função das 3 épocas).

In [ ]:
def linha(s):
    rep = ev.report(s, val["codigos"], val["visto_no_treino"])
    return {
        "top-5 código": rep.loc[("todos", "codigo"), "top5"],
        "top-5 classe": rep.loc[("todos", "classe"), "top5"],
        "top-5 código (novo)": rep.loc[("texto novo", "codigo"), "top5"],
        "top-5 classe (novo)": rep.loc[("texto novo", "classe"), "top5"],
        "top-1 código": rep.loc[("todos", "codigo"), "top1"],
        "top-1 classe": rep.loc[("todos", "classe"), "top1"],
        "top-1 código (novo)": rep.loc[("texto novo", "codigo"), "top1"],
        "top-1 classe (novo)": rep.loc[("texto novo", "classe"), "top1"],
    }


PCT = ["top-5 código", "top-5 classe", "top-5 código (novo)", "top-5 classe (novo)",
       "top-1 código", "top-1 classe", "top-1 código (novo)", "top-1 classe (novo)"]
tab = {}
for m in MODELOS:
    t = pd.DataFrame({o: linha(s) for o, s in por[m].items()}).T
    t.index = pd.MultiIndex.from_tuples(t.index, names=["taxa", "epoca", "tau"])
    t["perda validação (info)"] = [perdas[m][(o[0], o[1])] for o in t.index]
    t["horas de GPU (treino)"] = [custos[m, o[0]]["horas_gpu"] for o in t.index]
    t["custo $ (treino)"] = [custos[m, o[0]]["dolares"] for o in t.index]
    tab[m] = t.sort_values("top-5 código", ascending=False, kind="stable")  # empates pela ordem da grelha
    vis = tab[m].copy()
    vis.index = [nome_opcao(o) + ("  ← em vigor" if o == EM_VIGOR else "") for o in vis.index]
    print(f"\n{m}{'  (CONTROLO FALHOU: sem conclusões)' if m in FALHOU else ''}")
    display(vis.style.format({c: "{:.2%}" for c in PCT}
                             | {"perda validação (info)": "{:.4f}", "horas de GPU (treino)": "{:.2f}",
                                "custo $ (treino)": "{:.2f}"}, na_rep="NaN"))

## A decisão

A regra, escrita antes de correr (a mesma do E4-S1): o vencedor é a opção com o melhor top-5 no **código completo**, desde que não perca o top-5 na **classe** (métrica de guarda) e desde que bata a opção em vigor (taxa 2e-5, época 3, τ = 0,25: a do primeiro treino) pela regra de comparação do E4-T1. Em caso de empate, fica a opção em vigor.

**Bootstrap emparelhado**, com um exemplo: sorteiam-se 1 000 amostras de 23 945 contratos com reposição (o mesmo contrato pode sair duas vezes); em cada amostra calcula-se o top-5 das duas opções **nos mesmos contratos**. Se a melhor ganhar em pelo menos 95% das amostras, a diferença **conta**; senão, é empate. Faz-se só o bootstrap de que a regra precisa (melhor contra em vigor), mais os 3 concorrentes mais próximos para contexto: com as 27 seria cerca de meio minuto por par.

Depois da escolha: se o vencedor estiver num **extremo** da lista (a taxa mais baixa ou mais alta, o τ mais baixo ou mais alto, ou a última época), o melhor valor pode estar fora da grelha, e isso escreve-se como limitação (como no E4-S4).

In [ ]:
def duelo(a, b):
    res = {}
    for medida in ("codigo", "classe"):
        ra = cmp.weighted_mean(ev.hits(a, val["codigos"], medida, ev.TOP_K))
        rb = cmp.weighted_mean(ev.hits(b, val["codigos"], medida, ev.TOP_K))
        res[medida] = cmp.paired_bootstrap(ra, rb, len(val))
    return res, cmp.choose(res["codigo"], res["classe"], default="a")


ESCOLHIDO = {}
for m in MODELOS:
    t = tab[m]
    print("=" * 100)
    print(f"{m}{'  (CONTROLO FALHOU: o que se segue NÃO é uma conclusão)' if m in FALHOU else ''}")
    melhor = t["top-5 código"].idxmax()
    print(f"melhor na validação (top-5 código): {nome_opcao(melhor)} ({t.loc[melhor, 'top-5 código']:.2%})")
    print(f"em vigor:                           {nome_opcao(EM_VIGOR)} ({t.loc[EM_VIGOR, 'top-5 código']:.2%})")
    if melhor == EM_VIGOR:
        escolhido = EM_VIGOR
        print("  o melhor é a própria opção em vigor: não há comparação a fazer")
    else:
        res, escolha = duelo(por[m][EM_VIGOR], por[m][melhor])
        for medida, r in res.items():
            print(f"  {medida:7s} em vigor: {r['a']:.2%}  melhor: {r['b']:.2%}  diferença {r['b'] - r['a']:+.2%}  "
                  f"melhor ganha em {r['b_melhor']:.0%} das amostras (conta: {r['conta']})")
        escolhido = melhor if escolha == "b" else EM_VIGOR
    ESCOLHIDO[m] = escolhido
    print(f"-> fica {nome_opcao(escolhido)}")

    ext = gb.nos_extremos(escolhido, TAXAS, EPOCAS, TAUS)
    if ext:
        print("LIMITAÇÃO (extremo da lista): " + "; ".join(ext) + ". O melhor valor pode estar fora da grelha.")
    print(f"LIMITAÇÃO (viés de seleção): o vencedor saiu de {len(t)} opções; o seu número fica otimista "
          "e confirma-se no teste, na fase 5.")

    print("\nOs 3 concorrentes mais próximos (distintos do em vigor), contra o em vigor:")
    for o in [o for o in t.index if o != EM_VIGOR][:3]:
        r = duelo(por[m][EM_VIGOR], por[m][o])[0]["codigo"]
        print(f"  {nome_opcao(o)}: código {r['b']:.2%} vs {r['a']:.2%} (diferença {r['b'] - r['a']:+.2%}, conta={r['conta']})")

    print("\nMelhor top-5 no código completo por taxa:")
    for taxa in TAXAS:
        sub = t.xs(taxa, level="taxa")["top-5 código"]
        ep, tau = sub.idxmax()
        print(f"  taxa {gb.taxa_txt(taxa)}: {sub.max():.2%} (época {ep}, τ {tau:g})")

## O escolhido contra a opção em vigor

A avaliação completa (código, classe e divisão; top-1, top-5 e F1 macro; todos os contratos e texto novo) do escolhido de cada modelo, ao lado da opção em vigor. Se o escolhido for a própria opção em vigor, as duas colunas são iguais.

In [ ]:
for m in MODELOS:
    lado = pd.concat({"escolhido": ev.report(por[m][ESCOLHIDO[m]], val["codigos"], val["visto_no_treino"]),
                      "em vigor": ev.report(por[m][EM_VIGOR], val["codigos"], val["visto_no_treino"])}, axis=1)
    print(f"\n{m}: escolhido = {nome_opcao(ESCOLHIDO[m])}")
    display(lado.style.format("{:.1%}", subset=[c for c in lado.columns if c[1] in ("top1", "top5")])
            .format("{:.4f}", subset=[c for c in lado.columns if c[1] == "f1_macro"])
            .format("{:,.0f}", subset=[c for c in lado.columns if c[1] == "contratos"]))

## Sementes do vencedor

A **semente** fixa os números aleatórios de um treino: a ordem dos exemplos em cada época, os pesos iniciais da camada de classificação e o *dropout*. Com outra semente o mesmo treino dá outro modelo, e o top-5 muda um pouco. O bootstrap mede a incerteza por causa **dos contratos** da validação, não por causa da semente. Por isso o vencedor de cada modelo volta a treinar-se com as sementes do YAML (1 e 2), com a mesma taxa e as mesmas 3 épocas, e avalia-se na mesma época e com o mesmo τ.

Esta parte corre de novo depois de as sementes voltarem do pod. Enquanto as pastas não existirem, diz só que faltam.

In [ ]:
for m in MODELOS:
    taxa, ep, tau = ESCOLHIDO[m]
    sementes = [EXP["semente_grelha"]] + EXP["sementes_extra"]
    pastas = {s: pasta(m, taxa, s) for s in sementes}
    faltam = [s for s, p in pastas.items() if not (p / gb.TREINO_JSON).exists()]
    print(f"\n{m}: vencedor {nome_opcao(ESCOLHIDO[m])}, sementes {sementes}")
    if faltam:
        print(f"  ainda faltam as sementes {faltam} ({', '.join(pastas[s].name for s in faltam)}): "
              "esta célula corre de novo depois de virem do pod")
        continue
    linhas = {}
    for s, p in pastas.items():
        resumo = json.loads((p / gb.TREINO_JSON).read_text(encoding="utf-8"))
        for a in gb.verificar_treino(resumo, p.name, modelo=m, taxa=taxa, semente=s, epocas=EXP["epocas_max"],
                                     n_min=EXP["n_min"], taus=TAUS, validacao_sha256=VAL_SHA):
            print("  AVISO:", p.name, a)
        r = linha(gb.ler_sugestoes(gb.ficheiro_sugestoes(p, ep, tau), val["idcontrato"]))
        c = gb.custo(resumo, gb.ler_pod_info(p))
        linhas[f"semente {s}"] = {k: r[k] for k in ("top-5 código", "top-5 classe", "top-5 código (novo)",
                                                     "top-5 classe (novo)")} | {"custo $ (treino)": c["dolares"]}
    sem = pd.DataFrame(linhas).T
    met = [c for c in sem.columns if c.startswith("top-5")]
    resumo_sem = pd.DataFrame({"média": sem[met].mean(), "mínimo": sem[met].min(), "máximo": sem[met].max(),
                               "amplitude (pontos)": 100 * (sem[met].max() - sem[met].min())}).T
    display(sem.style.format({c: "{:.2%}" for c in met} | {"custo $ (treino)": "{:.2f}"}, na_rep="NaN"))
    display(resumo_sem.style.format("{:.2%}").format("{:.2f}", subset=pd.IndexSlice[["amplitude (pontos)"], :]))

## Conclusão

*(A escrever depois de a grelha correr, com os números das células acima: o controlo de cada modelo, a opção escolhida, as limitações — extremos, viés de seleção com 27 opções, N não testado — e a variação entre as 3 sementes. Os números e a decisão vão para a secção 23+ do `qualidade_dados.md`.)*